# Dental X-ray dataset exploration

Loads image/mask pairs from `../dataset`, overlays the segmentation labels, and computes per-class statistics.

Kernel: **Python (endo)**. Deps: numpy, pillow, matplotlib, pandas.

In [ ]:
import os, random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

ROOT = os.path.abspath("..")
IMG_DIR = os.path.join(ROOT, "dataset", "image")
LBL_DIR = os.path.join(ROOT, "dataset", "label")

# class map from dataset/label distribution.txt (index 0 = background)
CLASS_NAMES = {0: "background"}
with open(os.path.join(ROOT, "dataset", "label distribution.txt")) as f:
    for line in f:
        line = line.strip()
        if line:
            idx, name = line.split(" ", 1)
            CLASS_NAMES[int(idx)] = name
N_CLASSES = max(CLASS_NAMES) + 1
COLORS = plt.get_cmap("tab10")(np.arange(N_CLASSES))[:, :3]
CLASS_NAMES

In [ ]:
# image <-> label pairing (same filename stem, .jpg vs .png)
stems = sorted(p[:-4] for p in os.listdir(IMG_DIR) if p.endswith(".jpg"))
missing = [s for s in stems if not os.path.exists(os.path.join(LBL_DIR, s + ".png"))]
print(f"{len(stems)} pairs, {len(missing)} masks missing")
assert not missing

In [ ]:
def load(stem):
    """Return (grayscale image uint8 HxW, class-index mask uint8 HxW)."""
    img = np.array(Image.open(os.path.join(IMG_DIR, stem + ".jpg")).convert("L"))
    mask = np.array(Image.open(os.path.join(LBL_DIR, stem + ".png")))  # palette-indexed -> 2D
    assert mask.ndim == 2, f"{stem}: mask is not indexed ({mask.shape})"
    return img, mask

def overlay(img, mask, alpha=0.5):
    rgb = np.stack([img] * 3, -1).astype(float) / 255
    for c in range(1, N_CLASSES):
        m = mask == c
        rgb[m] = (1 - alpha) * rgb[m] + alpha * COLORS[c]
    return np.clip(rgb, 0, 1)

LEGEND = [Patch(facecolor=COLORS[c], label=CLASS_NAMES[c]) for c in range(1, N_CLASSES)]

# sanity check on one pair
_img, _mask = load(stems[0])
print(stems[0], _img.shape, "classes present:", sorted(np.unique(_mask)))

## Overlay a few random samples

In [ ]:
random.seed(0)
sample = random.sample(stems, 6)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, stem in zip(axes.flat, sample):
    img, mask = load(stem)
    ax.imshow(overlay(img, mask))
    ax.set_title(stem, fontsize=9)
    ax.axis("off")
fig.legend(handles=LEGEND, loc="lower center", ncol=5, fontsize=9)
fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
# side-by-side image / mask / overlay for one case
stem = sample[0]
img, mask = load(stem)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("image")
ax[1].imshow(mask, cmap="tab10", vmin=0, vmax=N_CLASSES - 1); ax[1].set_title("mask")
ax[2].imshow(overlay(img, mask)); ax[2].set_title("overlay")
for a in ax:
    a.axis("off")
fig.legend(handles=LEGEND, loc="lower center", ncol=5, fontsize=9)
fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

## Per-class statistics

Computed on a random sample (bump `SAMPLE_N` to `len(stems)` for the full set — ~a few minutes).

In [ ]:
SAMPLE_N = 400

random.seed(1)
subset = random.sample(stems, min(SAMPLE_N, len(stems)))

px = np.zeros(N_CLASSES, dtype=np.int64)        # total pixels per class
img_count = np.zeros(N_CLASSES, dtype=np.int64) # images containing the class
sizes = []
for stem in subset:
    _, mask = load(stem)
    sizes.append(mask.shape)
    vals, counts = np.unique(mask, return_counts=True)
    px[vals] += counts
    img_count[vals] += 1

stats = pd.DataFrame({
    "class": [CLASS_NAMES[i] for i in range(N_CLASSES)],
    "pixels": px,
    "pixel_share_%": (100 * px / px.sum()).round(3),
    "images_present": img_count,
    "image_freq_%": (100 * img_count / len(subset)).round(1),
})
stats

In [ ]:
fg = stats.iloc[1:]  # drop background
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

ax[0].barh(fg["class"], fg["pixel_share_%"], color=COLORS[1:])
ax[0].set_xscale("log")
ax[0].set_xlabel("pixel share % (log)")
ax[0].set_title("Class imbalance by area")
ax[0].invert_yaxis()

ax[1].barh(fg["class"], fg["image_freq_%"], color=COLORS[1:])
ax[1].set_xlabel("% of images containing the class")
ax[1].set_title("Class prevalence across images")
ax[1].invert_yaxis()

fig.tight_layout()
plt.show()

In [ ]:
# image orientation / size distribution
print(pd.Series([f"{w}x{h}" for h, w in sizes]).value_counts())

# how many distinct classes per image
cls_per_img = []
for stem in subset:
    _, mask = load(stem)
    cls_per_img.append(len(np.unique(mask)) - 1)  # exclude background
plt.figure(figsize=(7, 4))
plt.hist(cls_per_img, bins=range(0, N_CLASSES + 1), align="left", rwidth=0.8)
plt.xlabel("foreground classes per image")
plt.ylabel("images")
plt.title(f"n={len(subset)}")
plt.show()